In [2]:
"""
Knowledge Distillation — Script 3: Relation-Based
==================================================
Teacher : ResNet-50 (modified for CIFAR-10, pre-trained baseline)
Student : ResNet-18 (same CIFAR-10 modifications)
Dataset : CIFAR-10
Method  : Relation-based KD — match geometry between examples in embedding space

WHAT IS RELATION-BASED KD?
===========================
Instead of matching individual activation vectors, we match the STRUCTURAL
RELATIONSHIPS between examples: pairwise distances, triplet angles, or
kernel density manifolds. This is invariant to scale/rotation of the space
and excels at metric learning and retrieval tasks.

Three variants implemented:

1. RKD-D — Relational KD Distance (Park 2019)
   Match normalised pairwise Euclidean distances between embeddings:
     ψ(a,b) = ||f(a) - f(b)|| / μ   (μ = mean pairwise distance in batch)
     L_D = Σ_{(i,j)} ℓ_Huber(ψ_teacher(i,j), ψ_student(i,j))

2. RKD-A — Relational KD Angle (Park 2019)
   Match triplet angles — second-order geometry:
     cos∠(i,j,k) = <e_ij/||e_ij||, e_kj/||e_kj||>  where e_ij = f(i)-f(j)
     L_A = Σ_{(i,j,k)} ℓ_Huber(cos∠_t(i,j,k), cos∠_s(i,j,k))

3. RKD-D+A — Combined (typically best)
   L = CE + lambda_D * L_D + lambda_A * L_A

All use the penultimate embedding (avgpool output before FC) as the representation.

Sweeps:
  A — variant    : [rkd_d, rkd_a, rkd_da]         (lambda=25/50, cosine, 10ep)
  B — lambda     : [1, 10, 25, 50, 100]             (best variant from A)
  C — embed layer: [layer3_pool, layer4_pool]        (where to hook embeddings)

Output : __3__KD_relation_based.json
"""

import os, sys, json, time, copy, tempfile, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from torch.utils.data import DataLoader
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

warnings.filterwarnings("ignore")

# ── Device ─────────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[init] PyTorch {torch.__version__}", flush=True)
print(f"[init] Device  : {DEVICE}", flush=True)
if DEVICE.type == "cuda":
    print(f"[init] GPU     : {torch.cuda.get_device_name(0)}", flush=True)

# ── Config ─────────────────────────────────────────────────────────────────────
TEACHER_MODEL_PATH    = "../__2__baseline_resnet50_cifar10.pth"
BASELINE_METRICS_PATH = "../__2__baseline_metrics.json"
OUTPUT_JSON           = "__3__KD_relation_based_v2.json"

BATCH_SIZE  = 128 if DEVICE.type == "cuda" else 64
NUM_WORKERS = 0
PIN_MEMORY  = False
NUM_CLASSES = 10
KD_EPOCHS   = 10
USE_AMP     = (DEVICE.type == "cuda")

CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD  = (0.2023, 0.1994, 0.2010)

print(f"[init] AMP: {USE_AMP}", flush=True)


# ── Model builders ─────────────────────────────────────────────────────────────
def build_resnet50_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet50(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def build_resnet18_cifar(num_classes: int = 10) -> nn.Module:
    model = models.resnet18(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, num_classes)
    return model

def load_teacher(path: str) -> nn.Module:
    print(f"[load] Loading teacher from {path} ...", flush=True)
    model = build_resnet50_cifar(NUM_CLASSES)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    model.eval().to(DEVICE)
    print("[load] Teacher OK", flush=True)
    return model


# ── Embedding extractor ─────────────────────────────────────────────────────────
class EmbeddingExtractor(nn.Module):
    """
    Extracts the penultimate embedding (after avgpool, before fc).
    Returns (logits, embedding).
    """
    def __init__(self, model: nn.Module):
        super().__init__()
        self.model     = model
        self._embed    = None
        self._hook     = self.model.avgpool.register_forward_hook(
            lambda m, inp, out: setattr(self, "_embed",
                                        out.view(out.size(0), -1))
        )

    def forward(self, x):
        logits = self.model(x)
        return logits, self._embed

    def remove_hook(self):
        self._hook.remove()


# ── RKD Loss functions ──────────────────────────────────────────────────────────
def huber_loss(x: torch.Tensor, y: torch.Tensor,
               delta: float = 1.0) -> torch.Tensor:
    """Element-wise Huber (smooth-L1) loss, mean-reduced."""
    return F.huber_loss(x, y, delta=delta, reduction="mean")


def rkd_distance_loss(t_emb: torch.Tensor,
                      s_emb: torch.Tensor) -> torch.Tensor:
    """
    RKD-D: match normalised pairwise distances.

    ψ(a,b) = ||f(a)-f(b)|| / μ
    L_D = Σ_{(i,j)} ℓ_Huber(ψ_t(i,j), ψ_s(i,j))

    Uses all N*(N-1)/2 unique pairs in the batch.
    """
    # Pairwise L2 distances: (B, B)
    t_d = torch.cdist(t_emb, t_emb, p=2)
    s_d = torch.cdist(s_emb, s_emb, p=2)

    # Normalise by mean distance (excluding diagonal)
    B = t_emb.size(0)
    mask = ~torch.eye(B, dtype=torch.bool, device=t_emb.device)
    mu_t = t_d[mask].mean().clamp(min=1e-6)
    mu_s = s_d[mask].mean().clamp(min=1e-6)

    t_d_norm = t_d[mask] / mu_t
    s_d_norm = s_d[mask] / mu_s

    return huber_loss(s_d_norm, t_d_norm.detach())


def rkd_angle_loss(t_emb: torch.Tensor,
                   s_emb: torch.Tensor) -> torch.Tensor:
    """
    RKD-A: match triplet angles.

    For each triplet (i, j=anchor, k), compute cosine of angle at j:
      cos∠(i,j,k) = <e_ij/||e_ij||, e_kj/||e_kj||>
      where e_ij = f(i) - f(j)

    For efficiency, vectorise over the batch:
      We compute N^2 difference vectors and sample a random subset.
    """
    B = t_emb.size(0)

    # For efficiency, use a vectorised approach:
    # Expand to all pairs (i,j): diff[i,j] = emb[i] - emb[j]
    t_e = t_emb.unsqueeze(0) - t_emb.unsqueeze(1)   # (B, B, D)
    s_e = s_emb.unsqueeze(0) - s_emb.unsqueeze(1)

    # Normalize
    t_e = F.normalize(t_e, p=2, dim=2)
    s_e = F.normalize(s_e, p=2, dim=2)

    # For each anchor j, cosine between vectors to i and k:
    # cos_angle[j, i, k] = dot(t_e[i,j], t_e[k,j])
    # Vectorised: (B, B, D) x (B, D, B) → (B, B, B) — too large for B=128
    # Use a subset: for each anchor j, pick first B pairs of (i,k)
    # Actually compute (B, B) angle matrix for each anchor via batch matmul:
    # t_angles[j] = t_e[:,j,:] @ t_e[:,j,:].T  → (B,B) angles for anchor j
    # We collapse to (B*B, B*B) slice — use random J anchors for memory safety

    max_anchors = min(B, 32)   # limit triplets for memory
    idx_j = torch.randperm(B, device=t_emb.device)[:max_anchors]

    t_vecs = t_e[:, idx_j, :]     # (B, J, D)  difference to J anchors
    s_vecs = s_e[:, idx_j, :]

    # (B, J, D) x (D, B, J) — use einsum for (B, J, B) angle matrix
    t_angles = torch.einsum("ijd,kjd->ijk", t_vecs, t_vecs)   # (B,J,B)
    s_angles = torch.einsum("ijd,kjd->ijk", s_vecs, s_vecs)

    return huber_loss(s_angles, t_angles.detach())


# ── Data ────────────────────────────────────────────────────────────────────────
def get_train_loader():
    transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=True,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=BATCH_SIZE, shuffle=True,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

def get_test_loader():
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(CIFAR_MEAN, CIFAR_STD),
    ])
    ds = torchvision.datasets.CIFAR10(root="../data", train=False,
                                       download=True, transform=transform)
    return DataLoader(ds, batch_size=512, shuffle=False,
                      num_workers=NUM_WORKERS, pin_memory=PIN_MEMORY)

# ── Helpers ─────────────────────────────────────────────────────────────────────
def model_size_mb(model: nn.Module) -> float:
    with tempfile.NamedTemporaryFile(suffix=".pth", delete=False) as f:
        tmp = f.name
    try:
        torch.save(model.state_dict(), tmp)
        return os.path.getsize(tmp) / 1024 ** 2
    finally:
        os.remove(tmp)

def param_count(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters())

def sync():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

@torch.no_grad()
def evaluate(model: nn.Module, loader: DataLoader, device: torch.device) -> dict:
    model.eval()
    preds, labels = [], []
    for inputs, lbls in loader:
        inputs = inputs.to(device, non_blocking=True)
        out = model(inputs)
        logits = out[0] if isinstance(out, tuple) else out
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_pred, y_true = np.array(preds), np.array(labels)
    return {
        "accuracy" : float(accuracy_score(y_true, y_pred)),
        "precision": float(precision_score(y_true, y_pred, average="macro", zero_division=0)),
        "recall"   : float(recall_score(y_true, y_pred, average="macro", zero_division=0)),
        "f1"       : float(f1_score(y_true, y_pred, average="macro", zero_division=0)),
    }

def measure_inference_ms(model: nn.Module, device: torch.device, runs: int = 30) -> float:
    m = copy.deepcopy(model).to(device).eval()
    dummy = torch.randn(128, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(5): m(dummy)
        sync()
        t0 = time.perf_counter()
        for _ in range(runs): m(dummy)
        sync()
    return (time.perf_counter() - t0) / runs * 1000


def measure_inference_detailed(model: nn.Module, device: torch.device) -> dict:
    model = copy.deepcopy(model).eval().to(device)
    is_gpu = device.type == "cuda"
 
    # ── Single image ──────────────────────────────────────────
    dummy_single = torch.randn(1, 3, 32, 32, device=device)
    with torch.no_grad():
        for _ in range(50): model(dummy_single)
    if is_gpu: torch.cuda.synchronize()
 
    times = []
    with torch.no_grad():
        for _ in range(500):
            if is_gpu: torch.cuda.synchronize()
            t0 = time.perf_counter()
            model(dummy_single)
            if is_gpu: torch.cuda.synchronize()
            times.append(time.perf_counter() - t0)
    single_ms = float(np.mean(times) * 1000)
 
    # ── Batch 128 ─────────────────────────────────────────────
    dummy_batch = torch.randn(128, 3, 32, 32, device=device)
    if is_gpu:
        start_ev = torch.cuda.Event(enable_timing=True)
        end_ev   = torch.cuda.Event(enable_timing=True)
        with torch.no_grad():
            for _ in range(10): model(dummy_batch)
        torch.cuda.synchronize()
        batch_times = []
        with torch.no_grad():
            for _ in range(100):
                start_ev.record()
                model(dummy_batch)
                end_ev.record()
                torch.cuda.synchronize()
                batch_times.append(start_ev.elapsed_time(end_ev))
        batch_ms = float(np.mean(batch_times))
    else:
        with torch.no_grad():
            for _ in range(5): model(dummy_batch)
        t0 = time.perf_counter()
        with torch.no_grad():
            for _ in range(20): model(dummy_batch)
        batch_ms = (time.perf_counter() - t0) / 20 * 1000
 
    return {
        "single_img_gpu_ms"  : round(single_ms, 4),
        "batch128_gpu_ms"    : round(batch_ms, 4),
        "per_img_gpu_ms"     : round(batch_ms / 128, 4),
        "throughput_imgs_sec": round(128 / (batch_ms / 1000), 1),
        "timing_method"      : "CUDA events + torch.cuda.synchronize()",
    }
 
def compute_flops(model: nn.Module) -> float:
    from thop import profile as thop_profile
    m = copy.deepcopy(model).cpu().eval()
    dummy = torch.randn(1, 3, 32, 32)
    try:
        macs, _ = thop_profile(m, inputs=(dummy,), verbose=False)
        return round(float(macs * 2 / 1e9), 4)
    except Exception as e:
        print(f"      [flops] WARNING: thop failed ({e})", flush=True)
        return None
 

# ══════════════════════════════════════════════════════════════════════════════
# Core training loop
# ══════════════════════════════════════════════════════════════════════════════
def run_relation_kd(
    teacher_raw: nn.Module,
    train_loader: DataLoader,
    test_loader: DataLoader,
    baseline_acc: float,
    variant: str    = "rkd_da",    # "rkd_d" | "rkd_a" | "rkd_da"
    lambda_d: float = 25.0,        # weight for distance loss
    lambda_a: float = 50.0,        # weight for angle loss
    lr: float       = 0.05,
    n_epochs: int   = KD_EPOCHS,
    sweep_name: str = "variant",
) -> dict:
    tag = f"{variant}/λd={lambda_d}/λa={lambda_a}"
    print(f"\n  ┌─ [{tag}]", flush=True)

    try:
        student_raw = build_resnet18_cifar(NUM_CLASSES).to(DEVICE)

        teacher = EmbeddingExtractor(teacher_raw).to(DEVICE)
        student = EmbeddingExtractor(student_raw).to(DEVICE)
        teacher.eval()

        optimizer = torch.optim.SGD(student.parameters(), lr=lr,
                                    momentum=0.9, weight_decay=5e-4, nesterov=True)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)
        scaler    = torch.amp.GradScaler(enabled=USE_AMP)

        if DEVICE.type == "cuda":
            torch.cuda.reset_peak_memory_stats()

        epoch_train_acc  = []
        epoch_rkd_loss   = []
        t_start = time.perf_counter()

        for epoch in range(n_epochs):
            student.train()
            correct = total = 0
            rkd_loss_sum = 0.0
            t_ep = time.perf_counter()

            for batch_idx, (inputs, targets) in enumerate(train_loader):
                inputs  = inputs.to(DEVICE, non_blocking=True)
                targets = targets.to(DEVICE, non_blocking=True)

                optimizer.zero_grad(set_to_none=True)

                with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                    with torch.no_grad():
                        _, t_emb = teacher(inputs)   # (B, 2048)

                    s_logits, s_emb = student(inputs) # logits, (B, 512)

                    ce_loss = F.cross_entropy(s_logits, targets)

                    # Compute relational losses
                    rkd_loss = torch.tensor(0.0, device=DEVICE)
                    if variant in ("rkd_d", "rkd_da"):
                        rkd_loss = rkd_loss + lambda_d * rkd_distance_loss(t_emb, s_emb)
                    if variant in ("rkd_a", "rkd_da"):
                        rkd_loss = rkd_loss + lambda_a * rkd_angle_loss(t_emb, s_emb)

                    loss = ce_loss + rkd_loss

                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()

                correct += (s_logits.detach().argmax(1) == targets).sum().item()
                total   += targets.size(0)
                rkd_loss_sum += rkd_loss.item() * targets.size(0)

                if (batch_idx + 1) % 100 == 0:
                    elapsed = time.perf_counter() - t_ep
                    done = batch_idx + 1
                    eta  = elapsed / done * (len(train_loader) - done)
                    print(f"      [epoch {epoch+1}/{n_epochs}] "
                          f"batch {done}/{len(train_loader)}  "
                          f"acc={correct/total:.4f}  "
                          f"rkd={rkd_loss_sum/total:.4f}  "
                          f"ETA={eta:.0f}s", flush=True)

            scheduler.step()
            acc = correct / total
            epoch_train_acc.append(round(acc, 6))
            epoch_rkd_loss.append(round(rkd_loss_sum / total, 6))
            sync()
            ep_time = time.perf_counter() - t_ep
            remaining = n_epochs - (epoch + 1)
            print(f"      [epoch {epoch+1}/{n_epochs}] DONE  "
                  f"acc={acc:.4f}  rkd={rkd_loss_sum/total:.4f}  "
                  f"time={ep_time:.1f}s  ETA_remaining={ep_time*remaining:.0f}s", flush=True)

        sync()
        train_time_s = time.perf_counter() - t_start

        peak_gpu_mb = (round(torch.cuda.max_memory_allocated() / 1024**2, 2)
                       if DEVICE.type == "cuda" else None)

        teacher.remove_hook()
        student.remove_hook()

        print("      [eval] Evaluating student ...", flush=True)
        metrics  = evaluate(student.model, test_loader, DEVICE)
        inf      = measure_inference_detailed(student.model, DEVICE)
        size_mb  = model_size_mb(student.model)
        acc_drop = baseline_acc - metrics["accuracy"]
        flops_G  = compute_flops(student.model)
        total_params = param_count(student.model)

        # ── Save model ────────────────────────────────────────────
        save_dir = "__3__saved_models_KD"
        os.makedirs(save_dir, exist_ok=True)
        save_path = f"{save_dir}/student_resnet18_{variant}_ld{int(lambda_d)}_la{int(lambda_a)}.pth"
        torch.save(student.model.state_dict(), save_path)
        print(f"      [{tag}] ✓ Saved → {save_path}", flush=True)
        
        
        print(f"  └─ Acc={metrics['accuracy']:.4f}  Drop={acc_drop:+.4f}  "
              f"Size={size_mb:.2f}MB  Single={inf['single_img_gpu_ms']:.2f}ms  "
              f"Throughput={inf['throughput_imgs_sec']:.0f}imgs/s  "
              f"Train={train_time_s:.1f}s", flush=True)
        
        return {
            "sweep"             : sweep_name,
            "variant"           : variant,
            "lambda_d"          : lambda_d,
            "lambda_a"          : lambda_a,
            "lr"                : lr,
            "lr_schedule"       : "cosine",
            "epochs"            : n_epochs,
            "train_device"      : str(DEVICE),
            "use_amp"           : USE_AMP,
            "train_time_s"      : round(train_time_s, 2),
            "epoch_train_acc"   : epoch_train_acc,
            "epoch_rkd_loss"    : epoch_rkd_loss,
            
            "accuracy"          : round(metrics["accuracy"],  6),
            "precision"         : round(metrics["precision"], 6),
            "recall"            : round(metrics["recall"],    6),
            "f1"                : round(metrics["f1"],        6),
            "accuracy_drop"     : round(acc_drop, 6),
            "total_params"      : total_params,
            "size_mb"           : round(size_mb, 4),
            "flops_G"           : flops_G,
            "inference_ms"      : inf,
            "student_params"    : param_count(student.model),
            
            "peak_gpu_memory_mb": peak_gpu_mb,
            "description"       : (
                f"Relation-Based KD ({variant}, λd={lambda_d}, λa={lambda_a}, "
                f"cosine, epochs={n_epochs})"
            ),
            "status": "ok",
        }

    except Exception as e:
        import traceback
        print(f"  └─ FAILED: {e}", flush=True)
        traceback.print_exc()
        return {
            "sweep": sweep_name, "variant": variant,
            "lambda_d": lambda_d, "lambda_a": lambda_a,
            "status": "failed", "error": str(e),
            "accuracy": None, "accuracy_drop": None,
        }


# ── Main ────────────────────────────────────────────────────────────────────────
def main():
    print(f"\n{'='*65}")
    print("  Relation-Based Knowledge Distillation")
    print("  Teacher: ResNet-50 (modified)  →  Student: ResNet-18 (modified)")
    print(f"  Device: {DEVICE}  |  Epochs: {KD_EPOCHS}  |  Batch: {BATCH_SIZE}")
    print(f"{'='*65}\n")

    with open(BASELINE_METRICS_PATH) as f:
        baseline = json.load(f)
    baseline_acc = baseline["accuracy"]
    print(f"  Baseline teacher acc: {baseline_acc:.4f}\n", flush=True)

    teacher = load_teacher(TEACHER_MODEL_PATH)
    teacher_size_mb = model_size_mb(teacher)
    student_ref     = build_resnet18_cifar(NUM_CLASSES)

    print(f"  Teacher — Size: {teacher_size_mb:.2f} MB  Params: {param_count(teacher):,}")
    print(f"  Student — Size: {model_size_mb(student_ref):.2f} MB  Params: {param_count(student_ref):,}\n")

    train_loader = get_train_loader()
    test_loader  = get_test_loader()

    results = []

    # ── Sweep A: Variant ──────────────────────────────────────────────────────
    print("  ── Sweep A: Variant  (λd=25, λa=50, cosine) ──────────────────────", flush=True)
    sweep_a = []
    for variant in ["rkd_d", "rkd_a", "rkd_da"]:
        row = run_relation_kd(
            teacher, train_loader, test_loader, baseline_acc,
            variant=variant, lambda_d=25.0, lambda_a=50.0,
            lr=0.05, n_epochs=KD_EPOCHS, sweep_name="A_variant",
        )
        results.append(row)
        sweep_a.append(row)

    valid_a = [r for r in sweep_a if r.get("accuracy") is not None]
    if not valid_a:
        print("  ✗ Sweep A failed.", flush=True)
        return
    best_variant = max(valid_a, key=lambda r: r["accuracy"])["variant"]
    print(f"\n  Best variant: {best_variant}", flush=True)

    # ── Sweep B: Lambda (joint λd=λa=λ for simplicity) ───────────────────────
    print(f"\n  ── Sweep B: Lambda  (variant={best_variant}) ─────────────────────", flush=True)
    sweep_b = []
    for lam in [1.0, 10.0, 25.0, 50.0, 100.0]:
        row = run_relation_kd(
            teacher, train_loader, test_loader, baseline_acc,
            variant=best_variant,
            lambda_d=lam, lambda_a=lam * 2,  # keep A = 2*D ratio from Park 2019
            lr=0.05, n_epochs=KD_EPOCHS, sweep_name="B_lambda",
        )
        results.append(row)
        sweep_b.append(row)

    # ── Best overall ──────────────────────────────────────────────────────────
    valid = [r for r in results if r.get("accuracy") is not None]
    if not valid:
        print("  ✗ No valid results.", flush=True)
        return
    best = max(valid, key=lambda r: r["accuracy"])

    teacher_inf_ms = measure_inference_ms(teacher, DEVICE)

    report = {
        "method"       : "relation_based_kd",
        "description"  : "Relation-Based Knowledge Distillation — structural geometry matching",
        "teacher_arch" : "ResNet-50 (CIFAR-10 modified)",
        "student_arch" : "ResNet-18 (CIFAR-10 modified)",
        "dataset"      : "CIFAR-10",
        "train_device" : str(DEVICE),
        "use_amp"      : USE_AMP,
        "kd_epochs"    : KD_EPOCHS,
        "baseline"     : baseline,
        "teacher_info" : {
            "size_mb"     : round(teacher_size_mb, 4),
            "inference_ms": round(teacher_inf_ms, 4),
            "params"      : param_count(teacher),
        },
        "student_info" : {
            "size_mb"     : round(model_size_mb(student_ref), 4),
            "params"      : param_count(student_ref),
        },
        "best_config"  : {
            "variant"     : best["variant"],
            "lambda_d"    : best["lambda_d"],
            "lambda_a"    : best["lambda_a"],
            "accuracy"    : best["accuracy"],
            "accuracy_drop": best["accuracy_drop"],
            "size_mb"     : best["size_mb"],
            "inference_ms": best["inference_ms"],
        },
        "sweeps"       : {
            "A_variant"   : "rkd_d vs rkd_a vs rkd_da  (λd=25, λa=50, cosine)",
            "B_lambda"    : "λ in [1,10,25,50,100]  (best variant, cosine)",
        },
        "results"      : results,
    }

    with open(OUTPUT_JSON, "w") as f:
        json.dump(report, f, indent=2)

    print(f"\n  ✓ Saved → {OUTPUT_JSON}")
    print(f"  Best: variant={best['variant']} | λd={best['lambda_d']} | "
          f"λa={best['lambda_a']} | Acc={best['accuracy']:.4f} | "
          f"Drop={best['accuracy_drop']:+.4f}")


if __name__ == "__main__":
    main()

[init] PyTorch 2.12.0.dev20260324+cu128
[init] Device  : cuda
[init] GPU     : NVIDIA GeForce RTX 5070 Laptop GPU
[init] AMP: True

  Relation-Based Knowledge Distillation
  Teacher: ResNet-50 (modified)  →  Student: ResNet-18 (modified)
  Device: cuda  |  Epochs: 10  |  Batch: 128

  Baseline teacher acc: 0.9320

[load] Loading teacher from ../__2__baseline_resnet50_cifar10.pth ...
[load] Teacher OK
  Teacher — Size: 90.03 MB  Params: 23,520,842
  Student — Size: 42.70 MB  Params: 11,173,962

  ── Sweep A: Variant  (λd=25, λa=50, cosine) ──────────────────────

  ┌─ [rkd_d/λd=25.0/λa=50.0]
      [epoch 1/10] batch 100/391  acc=0.3250  rkd=0.7701  ETA=22s
      [epoch 1/10] batch 200/391  acc=0.4014  rkd=0.7209  ETA=14s
      [epoch 1/10] batch 300/391  acc=0.4535  rkd=0.6885  ETA=7s
      [epoch 1/10] DONE  acc=0.4913  rkd=0.6650  time=28.6s  ETA_remaining=258s
      [epoch 2/10] batch 100/391  acc=0.6546  rkd=0.5535  ETA=21s
      [epoch 2/10] batch 200/391  acc=0.6707  rkd=0.5373  E